In [ ]:
!pip install -q transformers==4.46.3 accelerate datasets evaluate rouge_score sentencepiece

import torch
import os
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM
from google.colab import drive
import evaluate
from datasets import load_dataset
from tqdm.auto import tqdm

drive.mount('/content/drive')

In [ ]:
MODEL_NAME = "VietAI/vit5-large-vietnews-summarization"

print("Đang tải Tokenizer từ VietAI...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, legacy=False)

print("Đang tải mô hình gốc của tác giả (Float16)...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

In [ ]:
import json
from datasets import Dataset

NUM_SAMPLES = 10000
PATH_TO_TEST_JSON = "/content/drive/MyDrive/Final-Statistical-Learning/test_data.json"

with open(PATH_TO_TEST_JSON, 'r', encoding='utf-8') as f:
    test_data_list = json.load(f)

test_subset = Dataset.from_list(test_data_list)

def clean_data(batch):
    cleaned_articles = [doc.replace('_', ' ') for doc in batch["article"]]
    cleaned_abstracts = [abs.replace('_', ' ') for abs in batch["abstract"]]
    return {
        "article": cleaned_articles,
        "abstract": cleaned_abstracts
    }

test_subset = test_subset.map(clean_data, batched=True, batch_size=100)

In [9]:
import gc

# 1. Giải phóng bộ nhớ đệm CUDA trước khi chạy
if 'inputs' in locals(): del inputs
if 'outputs' in locals(): del outputs
gc.collect()
torch.cuda.empty_cache()

rouge_metric = evaluate.load("rouge")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

predictions = []
references = []

articles = [item["article"] for item in test_subset]
references = [item["abstract"] for item in test_subset]

# 2. GIẢM BATCH_SIZE xuống 2 để bảo vệ bộ nhớ GPU
BATCH_SIZE = 16

for i in tqdm(range(0, len(articles), BATCH_SIZE)):
    batch_articles = articles[i : i + BATCH_SIZE]
    input_texts = ["vietnews: " + art for art in batch_articles]

    inputs = tokenizer(
        input_texts,
        return_tensors="pt",
        max_length=1024,
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        # 3. Thay đổi cơ chế sinh chuỗi: bỏ num_beams=4 để tiết kiệm RAM CUDA
        outputs = model.generate(
            **inputs,
            max_length=200,
            do_sample=True,         # Sử dụng cơ chế lấy mẫu thay cho Beam Search
            top_k=50,
            top_p=0.92,
            temperature=0.7,
            no_repeat_ngram_size=2
        )

    pred_summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(pred_summaries)

    # Giải phóng biến trung gian ngay trong vòng lặp để tránh tích tụ bộ nhớ
    del inputs, outputs
    if i % 50 == 0:
        torch.cuda.empty_cache()

print("\n📊 Đang tính toán điểm ROUGE cho mô hình tác giả...")
results = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=False)

print("="*50)
print("🏆 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH VIETAI (ROUGE SCORE)")
print("="*50)
print(f"📌 ROUGE-1 (Độ trùng lặp từng từ):     {results['rouge1'] * 100:.2f} %")
print(f"📌 ROUGE-2 (Độ trùng lặp cụm 2 từ):    {results['rouge2'] * 100:.2f} %")
print(f"📌 ROUGE-L (Độ trùng lặp chuỗi dài):   {results['rougeL'] * 100:.2f} %")
print(f"📌 ROUGE-Lsum (Điểm tổng hợp):         {results['rougeLsum'] * 100:.2f} %")
print("="*50)

  0%|          | 0/625 [00:00<?, ?it/s]


📊 Đang tính toán điểm ROUGE cho mô hình tác giả...
🏆 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH VIETAI (ROUGE SCORE)
📌 ROUGE-1 (Độ trùng lặp từng từ):     63.13 %
📌 ROUGE-2 (Độ trùng lặp cụm 2 từ):    33.87 %
📌 ROUGE-L (Độ trùng lặp chuỗi dài):   43.13 %
📌 ROUGE-Lsum (Điểm tổng hợp):         43.12 %
